# Average RMSD of the whole CESA1 complex

Computes backbone RMSD vs the traj1 reference frame for each of the 8 replicas,
then aggregates them into a single average RMSD curve.

**Inputs:**
- Reference: `backbone.pdb`
- Per-replica backbone trajectories: `traj1.xtc` → `traj8.xtc`

**Outputs:**
- `rmsd_whole_complex_CESA1.csv` — per-frame RMSD per replica
- `rmsd_whole_complex_CESA1_summary.csv` — mean / std / min / max per replica + overall
- `rmsd_whole_complex_CESA1.png` — RMSD vs time plot

In [ ]:
# ============================================================
# CELL 1 — Install MDAnalysis if needed
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'MDAnalysis', '-q'])
print('MDAnalysis ready')

In [ ]:
# ============================================================
# CELL 2 — Imports (exactly like collaborators)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.analysis import rms

print('MDAnalysis', mda.__version__)

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================

BASE_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/result_CESA1_ClusPro/CESA1_prod/Backbone_xtc')

REPLICAS = ['traj1', 'traj2', 'traj3', 'traj4', 'traj5', 'traj6', 'traj7', 'traj8']
REF_PDB  = BASE_DIR / 'backbone.pdb'       # one fixed reference for ALL replicas
TRAJ_FMT = '{R}.xtc'                        # traj1.xtc, traj2.xtc, etc.
ATOM_SEL = 'backbone'                       # MDAnalysis selection for RMSD

# Output files
OUT_DIR        = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/result_CESA1_ClusPro/run')
OUT_CSV_FRAMES = OUT_DIR / 'rmsd_whole_complex_CESA1.csv'
OUT_CSV_STATS  = OUT_DIR / 'rmsd_whole_complex_CESA1_summary.csv'
OUT_PNG        = OUT_DIR / 'rmsd_whole_complex_CESA1.png'

# Verify all files exist
assert REF_PDB.exists(), f'Reference PDB not found: {REF_PDB}'
for R in REPLICAS:
    p = BASE_DIR / TRAJ_FMT.format(R=R)
    assert p.exists(), f'Missing trajectory: {p}'
    print(f'  {R}: {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')
print(f'Reference: {REF_PDB.name}')

In [ ]:
# ============================================================
# CELL 4 — Per-replica RMSD
#
# Key difference from previous script:
# ALL replicas are compared against the SAME fixed reference
# (backbone.pdb) — not each replica's own frame 0
# This makes replicas directly comparable to each other
# RMSD is returned in Å by MDAnalysis
# ============================================================

results = {}   # replica name -> (time_ns, rmsd_A)

for R in REPLICAS:
    traj = BASE_DIR / TRAJ_FMT.format(R=R)
    print(f'-- {R}: loading {traj.name}')

    u   = mda.Universe(str(REF_PDB), str(traj))   # topology + trajectory
    ref = mda.Universe(str(REF_PDB))               # fixed reference structure

    rmsd_analysis = rms.RMSD(u, ref, select=ATOM_SEL).run()

    arr  = rmsd_analysis.results.rmsd   # columns: frame, time(ps), rmsd(Å)
    t_ns = arr[:, 1] / 1000.0           # ps → ns
    r_a  = arr[:, 2]                    # Å

    results[R] = (t_ns, r_a)

    print(f'   frames={len(arr)}  duration={t_ns[-1]-t_ns[0]:.1f} ns  '
          f'mean RMSD={r_a.mean():.3f} Å  std={r_a.std():.3f} Å')

In [ ]:
# ============================================================
# CELL 5 — Build per-frame DataFrame
# ============================================================

long_rows = []
for R, (t, r) in results.items():
    for ti, ri in zip(t, r):
        long_rows.append({'replica': R, 'time_ns': ti, 'rmsd_A': ri})

df_frames = pd.DataFrame(long_rows)
df_frames.to_csv(OUT_CSV_FRAMES, index=False)
print(f'Wrote {OUT_CSV_FRAMES.name}  ({len(df_frames):,} rows)')
df_frames.head()

In [ ]:
# ============================================================
# CELL 6 — Align replicas to common time axis
#           (intersection of all replicas — same as collaborators)
# ============================================================

# Common window = overlap between all replicas
t_lo = max(t[0]  for t, _ in results.values())
t_hi = min(t[-1] for t, _ in results.values())
dt   = min(np.median(np.diff(t)) for t, _ in results.values())
t_common = np.arange(t_lo, t_hi + dt/2, dt)

# Interpolate each replica onto common time axis
interp = {R: np.interp(t_common, t, r) for R, (t, r) in results.items()}
stack  = np.vstack([interp[R] for R in REPLICAS])   # shape (8, n_frames)

rmsd_mean = stack.mean(axis=0)
rmsd_std  = stack.std(axis=0)

print(f'Common time window: {t_lo:.1f} ns → {t_hi:.1f} ns  ({len(t_common)} points, dt={dt:.3f} ns)')

In [ ]:
# ============================================================
# CELL 7 — Summary table
# ============================================================

summary_rows = []
for R, (_, r) in results.items():
    summary_rows.append({
        'replica' : R,
        'n_frames': len(r),
        'mean_A'  : float(np.mean(r)),
        'std_A'   : float(np.std(r)),
        'min_A'   : float(np.min(r)),
        'max_A'   : float(np.max(r)),
    })

# Overall = mean of averaged curve on common time axis
summary_rows.append({
    'replica' : 'AVERAGE (across replicas)',
    'n_frames': len(t_common),
    'mean_A'  : float(rmsd_mean.mean()),
    'std_A'   : float(rmsd_mean.std()),
    'min_A'   : float(rmsd_mean.min()),
    'max_A'   : float(rmsd_mean.max()),
})

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_CSV_STATS, index=False)
print(f'Wrote {OUT_CSV_STATS.name}')
df_summary.round({'mean_A': 3, 'std_A': 3, 'min_A': 3, 'max_A': 3})

In [ ]:
# ============================================================
# CELL 8 — Plot (same style as collaborators)
# ============================================================

colors = {
    'traj1': '#1f77b4', 'traj2': '#2ca02c', 'traj3': '#d62728',
    'traj4': '#9467bd', 'traj5': '#8c564b', 'traj6': '#e377c2',
    'traj7': '#7f7f7f', 'traj8': '#bcbd22'
}

fig, ax = plt.subplots(figsize=(9, 4.5))

# Individual replicas
for R in REPLICAS:
    t, r = results[R]
    ax.plot(t, r, color=colors.get(R, 'grey'), lw=0.9, alpha=0.55, label=R)

# Average line
ax.plot(t_common, rmsd_mean, color='black', lw=1.6, label='mean across replicas')

# Std shading
ax.fill_between(
    t_common,
    rmsd_mean - rmsd_std,
    rmsd_mean + rmsd_std,
    color='black', alpha=0.15, label='±1 SD'
)

ax.set_xlabel('Time (ns)')
ax.set_ylabel('Backbone RMSD (Å)')
ax.set_title('CESA1/HvGlu3 complex — backbone RMSD vs reference')
ax.legend(loc='best', frameon=False, ncol=3, fontsize=9)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_PNG, dpi=200)
print(f'Wrote {OUT_PNG.name}')
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Headline number (same as collaborators)
# ============================================================

overall_mean = rmsd_mean.mean()
overall_std  = rmsd_mean.std()

print(f'Average RMSD of the whole complex (mean across {len(REPLICAS)} replicas, '
      f'over {t_common[0]:.1f}–{t_common[-1]:.1f} ns):')
print(f'    {overall_mean:.3f} ± {overall_std:.3f} Å')
